# Actividad 4A. Cuaderno Jupyter: Descriptores de textura, forma y color con reducción de dimensión
**Procesamiento Digital de Imágenes · MCDI · INFOTEC · 2027-1** · Unidad 4

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/egonzalez1989/pdi-mcdi/blob/main/plantillas/4A_Plantilla_Descriptores.ipynb)

| | |
|---|---|
| **Estudiante** | *Escribe aquí tu nombre completo* |
| **Tipo de actividad** | Individual |
| **Fecha límite** | Domingo 4 de abril de 2027, 23:59 hrs. |
| **Puntaje** | 25 puntos |

**Propósito.** Construir descriptores de textura, forma y color, combinarlos en un vector de características y explorar la separación entre clases con PCA.

**Producto esperado.** Una libreta Jupyter bien documentada que incluya:
- Descriptores de textura GLCM (contraste, energía, homogeneidad) y, opcionalmente, LBP.
- Descriptores de color (histogramas normalizados o momentos de color).
- Descriptores de forma (momentos de Hu) a partir de una versión binarizada de los objetos.
- Vector de características combinado y estandarizado, con proyección PCA en 2D por clase.
- Comentarios en Markdown sobre la información que captura cada grupo de descriptores.

### Instrucciones
1. Descarga el cuaderno plantilla 4A_Plantilla_Descriptores.ipynb. Elige un conjunto de al menos 20 imágenes o parches con al menos dos clases visualmente diferenciables (puedes usar la carpeta datos/texturas del repositorio) y registra la etiqueta de cada una.
2. Textura (GLCM): convierte a escala de grises, calcula la matriz de co-ocurrencia con al menos un par (distancia, ángulo) y extrae contraste, energía y homogeneidad. Opcional: histograma LBP uniforme.
3. Color: calcula histogramas por canal normalizados o momentos de color (media, varianza, asimetría) en RGB o HSV y comenta qué aportan frente a la textura.
4. Forma: a partir de una versión binarizada, calcula los momentos invariantes de Hu (en escala logarítmica) u otros descriptores sencillos (área, circularidad).
5. Vector de características: combina textura, color y forma en un vector por imagen, construye la matriz de diseño (tabla de pandas) y estandariza las columnas.
6. Reducción de dimensión: aplica PCA a 2 componentes, reporta la varianza explicada y grafica PC1 vs. PC2 con un color por clase. Opcional: compara con t-SNE.
7. Interpretación: discute si se observa separación entre clases, qué clases se confunden y qué descriptor agregarías o quitarías.
8. Guarda tu libreta como 4A_Nombre_Apellido.ipynb, expórtala a PDF con el mismo nombre y entrega ambos archivos.

### Lista de cotejo (así se evaluará tu trabajo)
| Criterio | Puntaje |
|---|---|
| Cálculo adecuado de descriptores de textura, color y forma. | 7 |
| Construcción del vector de características (estandarizado) y aplicación de reducción de dimensión. | 7 |
| Interpretación de los resultados (separación, similitudes, limitaciones). | 5 |
| Claridad en el código, visualizaciones y documentación en Markdown. | 6 |
| **Total** | **25** |

> 💡 Escribe tus interpretaciones en celdas Markdown como esta. Una figura sin interpretación no demuestra aprendizaje. Si usas fragmentos de código de otras fuentes (incluidos asistentes de IA), cítalos en la sección final.

In [ ]:
# Configuración del entorno. En Google Colab todas estas bibliotecas ya vienen instaladas.
import sys, platform
import numpy as np
import matplotlib.pyplot as plt
import cv2
import skimage
from skimage import data

plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 10, "image.cmap": "gray"})
np.random.seed(42)

def mostrar(imagenes, titulos=None, cols=None, cmap="gray", tam=3.2, ejes=False):
    """Muestra una lista de imágenes en una cuadrícula. Las imágenes a color deben estar en RGB."""
    if not isinstance(imagenes, (list, tuple)):
        imagenes = [imagenes]
    n = len(imagenes)
    cols = cols or min(n, 4)
    filas = int(np.ceil(n / cols))
    fig, axs = plt.subplots(filas, cols, figsize=(tam * cols, tam * filas), squeeze=False)
    for i, ax in enumerate(axs.flat):
        if i < n:
            im = imagenes[i]
            ax.imshow(im, cmap=None if im.ndim == 3 else cmap)
            if titulos:
                ax.set_title(titulos[i])
        if not ejes or i >= n:
            ax.axis("off")
    plt.tight_layout()
    plt.show()

print(f"Python {platform.python_version()} | NumPy {np.__version__} | OpenCV {cv2.__version__} | scikit-image {skimage.__version__}")

# Datos del curso: se buscan en la carpeta datos/ del repositorio (si lo clonaste) o se descargan.
import os, io, zipfile, urllib.request
def carpeta_datos(url="https://github.com/egonzalez1989/pdi-mcdi/archive/refs/heads/main.zip"):
    for d in ("datos", "../datos", "pdi-mcdi-main/datos"):
        if os.path.isdir(d):
            return d
    try:
        with urllib.request.urlopen(url, timeout=20) as r:
            zipfile.ZipFile(io.BytesIO(r.read())).extractall(".")
        return "pdi-mcdi-main/datos"
    except Exception as e:
        print("No se pudieron descargar los datos del curso:", e)
        return None
DATOS = carpeta_datos()
print("Carpeta de datos:", DATOS)

import glob
import pandas as pd
from scipy.stats import skew
from skimage import feature
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

## 1. Conjunto de imágenes
Organiza tus imágenes en carpetas, una por clase (`mis_datos/clase_a/*.jpg`, …). Por omisión se usan los parches de `datos/texturas` (ladrillo, pasto y grava).

In [ ]:
CARPETA = ""   # p. ej. "mis_datos"
if not CARPETA and DATOS:
    CARPETA = os.path.join(DATOS, "texturas")
df = pd.DataFrame([{"ruta": r, "clase": os.path.basename(os.path.dirname(r))}
                   for r in sorted(glob.glob(os.path.join(CARPETA, "*", "*")))
                   if r.lower().endswith((".png", ".jpg", ".jpeg", ".tif", ".bmp"))])
print(df.clase.value_counts() if not df.empty else "No se encontraron imágenes")

## 2. Descriptores de textura (GLCM; opcional LBP)

In [ ]:
def descriptores_textura(gris):
    """TODO: feature.graycomatrix + feature.graycoprops: contraste, energía y homogeneidad."""
    return {}

## 3. Descriptores de color

In [ ]:
def descriptores_color(img):
    """TODO: histogramas normalizados por canal o momentos de color (media, desviación, asimetría) en RGB o HSV.
    Si la imagen es en escala de grises, usa los momentos de intensidad."""
    return {}

## 4. Descriptores de forma

In [ ]:
def descriptores_forma(gris):
    """TODO: binariza (p. ej., Otsu) y calcula los 7 momentos de Hu en escala log: -sign(h)*log10(|h|)."""
    return {}

## 5. Vector de características y matriz de diseño

In [ ]:
filas = []
for _, r in df.iterrows():
    img = cv2.imread(r.ruta, cv2.IMREAD_UNCHANGED)
    gris = img if img.ndim == 2 else cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    filas.append({**descriptores_textura(gris), **descriptores_color(img), **descriptores_forma(gris), "clase": r.clase})
X = pd.DataFrame(filas)
X.head()
# TODO: separa las columnas numéricas y estandarízalas con StandardScaler

## 6. Reducción de dimensión con PCA

In [ ]:
# TODO: PCA(n_components=2); reporta pca.explained_variance_ratio_ y grafica PC1 vs PC2 coloreando por clase
# Opcional: compara con sklearn.manifold.TSNE

## 7. Interpretación
- ¿Se observa separación entre clases? ¿Cuáles se confunden y por qué?
- ¿Qué aporta cada grupo de descriptores?
- ¿Qué descriptor agregarías o quitarías?

## Referencias y uso de herramientas
*Lista en formato APA las fuentes consultadas. Si usaste asistentes de IA generativa, indica cuál, para qué y verifica que entiendes cada línea de código que entregas.*

---
**Antes de entregar:** `Entorno de ejecución → Reiniciar y ejecutar todo`, verifica que no haya errores y exporta a PDF (`Archivo → Imprimir → Guardar como PDF` o `jupyter nbconvert --to webpdf`).